In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, precision_recall_curve, average_precision_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# ====================== 随机种子 & 设备 ======================
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

LOCAL_MODEL_DIR = "./hf_models"
MODEL_PATH = os.path.join(LOCAL_MODEL_DIR, "roberta-base")

# ====================== 数据加载 ======================
df = pd.read_csv("mental_heath_unbanlanced.csv")
df = df.dropna(subset=["text", "status"])
df["text"] = df["text"].astype(str)

label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df["status"])

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df["text"].tolist(),
    df["label_encoded"].tolist(),
    test_size=0.3,
    random_state=42,
    stratify=df["label_encoded"]
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42,
    stratify=temp_labels
)

# ====================== Dataset ======================
class RiskRegressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=96):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        single_label = self.labels[idx]
        item = {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "single_label": torch.tensor(single_label, dtype=torch.long),
            "text": self.texts[idx]
        }
        return item

# ====================== 加载回归模型（与训练时一致） ======================
class DualRegressionModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_path, local_files_only=True)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        
        self.dep_head = nn.Linear(hidden, 1)
        self.sui_head = nn.Linear(hidden, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        cls = self.dropout(cls)
        
        dep_score = torch.sigmoid(self.dep_head(cls)).squeeze(-1)
        sui_score = torch.sigmoid(self.sui_head(cls)).squeeze(-1)
        
        return {"dep_score": dep_score, "sui_score": sui_score}

# ====================== 加载 checkpoint ======================
print("加载回归模型 checkpoint...")
model = DualRegressionModel(MODEL_PATH).to(DEVICE)
model.load_state_dict(torch.load("best_dual_regression.pt", map_location=DEVICE))
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

test_dataset = RiskRegressionDataset(test_texts, test_labels, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ====================== 推理获取分数 ======================
print("正在对测试集进行推理...")
test_sui_scores = []
test_dep_scores = []
test_true_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        single_labels = batch["single_label"].numpy()
        
        outputs = model(input_ids, attention_mask)
        
        test_sui_scores.extend(outputs["sui_score"].cpu().numpy())
        test_dep_scores.extend(outputs["dep_score"].cpu().numpy())
        test_true_labels.extend(single_labels)

test_sui_scores = np.array(test_sui_scores)
test_true_labels = np.array(test_true_labels)

# ====================== 高风险检测指标 ======================
high_risk_true = (test_true_labels == 3).astype(int)  # Suicidal 作为 positive

print("\n=== High-risk Detection Metrics (Regression-based) ===")
thresholds = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8]

print("阈值\tPrecision\tRecall\tF1\tAUC")
auc = roc_auc_score(high_risk_true, test_sui_scores)
for th in thresholds:
    high_risk_pred = (test_sui_scores >= th).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(high_risk_true, high_risk_pred, average='binary')
    print(f"{th:.2f}\t{p:.4f}\t{r:.4f}\t{f1:.4f}\t{auc:.4f}")

# ====================== Precision-Recall 曲线 ======================
precision, recall, pr_thresholds = precision_recall_curve(high_risk_true, test_sui_scores)
ap = average_precision_score(high_risk_true, test_sui_scores)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, marker='.', label=f'AP = {ap:.3f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve for High-risk Detection (Suicidal)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('pr_curve_high_risk.png', dpi=300, bbox_inches='tight')
plt.close()
print("\nPR 曲线已保存为 pr_curve_high_risk.png")

# ====================== 与分类模型的 Suicidal 类指标对比 ======================
print("\n=== 与分类模型的 Suicidal 类指标对比 ===")
print(f"{'模型类型':<35} {'Suicidal Recall':<15} {'Precision':<12} {'F1':<10}")
print("-" * 70)
print(f"Base Dual-Regression (thresh=0.7)      {0.8026:<15.4f} {0.7768:<12.4f} {0.7895:<10.4f}")
print(f"+ Multi-label + Severity               {0.9049:<15.4f} {0.6937:<12.4f} {0.7853:<10.4f}")
print(f"+ Class Weight                         {0.7485:<15.4f} {0.7781:<12.4f} {0.7630:<10.4f}")
print(f"+ Hierarchical                         {0.8633:<15.4f} {0.6984:<12.4f} {0.7721:<10.4f}")
print(f"+ Ordinal Regression                   {0.8210:<15.4f} {0.7510:<12.4f} {0.7844:<10.4f}")

print("\n实验完成！请查看生成的 pr_curve_high_risk.png 和控制台输出的阈值表。")


D:\SCI\torchGPU\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
加载回归模型 checkpoint...


Loading weights: 100%|██| 197/197 [00:00<00:00, 574.34it/s, Materializing param=encoder.layer.11.output.dense.weight]
RobertaModel LOAD REPORT from: ./hf_models\roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


正在对测试集进行推理...


100%|██████████████████████████████████████████████████████████████████████████████| 233/233 [00:22<00:00, 10.33it/s]



=== High-risk Detection Metrics (Regression-based) ===
阈值	Precision	Recall	F1	AUC
0.50	0.5384	0.9661	0.6915	0.9533
0.55	0.6575	0.9233	0.7681	0.9533
0.60	0.7030	0.8811	0.7821	0.9533
0.65	0.7416	0.8430	0.7891	0.9533
0.70	0.7768	0.8026	0.7895	0.9533
0.75	0.8116	0.7402	0.7743	0.9533
0.80	0.8509	0.6480	0.7357	0.9533

PR 曲线已保存为 pr_curve_high_risk.png

=== 与分类模型的 Suicidal 类指标对比 ===
模型类型                                Suicidal Recall Precision    F1        
----------------------------------------------------------------------
Base Dual-Regression (thresh=0.7)      0.8026          0.7768       0.7895    
+ Multi-label + Severity               0.9049          0.6937       0.7853    
+ Class Weight                         0.7485          0.7781       0.7630    
+ Hierarchical                         0.8633          0.6984       0.7721    
+ Ordinal Regression                   0.8210          0.7510       0.7844    

实验完成！请查看生成的 pr_curve_high_risk.png 和控制台输出的阈值表。
